# Predict thermal frames with the EDTC YOLOv5s weights

Simplest path from `model_weights/edtc_yolov5s.pt` to a few detections on Anti-UAV thermal frames,
using the existing scaffolding: `thesis.models.load_model` (torch.hub `custom`) +
`thesis.antiuav.dataset.AntiUAVFrames` (sqlite index -> frame paths + GT boxes).

Run top to bottom. The first run downloads the `ultralytics/yolov5` hub repo once
(`trust_repo=True` skips the interactive trust prompt); its runtime deps are in the
`dev` dependency group of `pyproject.toml` (`uv sync`).

Predictions run on the **thermal (infrared) frames only** - EDTC was trained on thermal;
the RGB stream is loaded by `AntiUAVFrames` but unused here. Thermal PNGs are stored as
3-channel (grayscale replicated), matching the model's `ch=3` input.

The checkpoint is a single-class YOLOv5s (`nc=1`, `names={0: 'uav'}`); thermal frames are
640x512, already a multiple of the model stride, so the hub model's AutoShape preprocessing
(letterbox + NMS) returns boxes in original 640x512 pixel coordinates.

In [ ]:
import torch

import thesis
from thesis.config import Config
from thesis.antiuav.dataset import AntiUAVFrames
from thesis.models import load_model
from thesis.paths import ANTI_UAV_DATA_DIR, MODEL_WEIGHTS_DIR, join

# dataset root + weights from thesis.paths (anchored at the repo root)
DATA_ROOT = ANTI_UAV_DATA_DIR
WEIGHTS = MODEL_WEIGHTS_DIR / "edtc_yolov5s.pt"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# load the local EDTC checkpoint through the hub 'custom' entry point
cfg = Config(model_name="yolov5s", model_yolo_weights=WEIGHTS)
model = load_model(cfg, DEVICE, train=False)

# unwrap AutoShape -> DetectMultiBackend -> DetectionModel for introspection
inner = model
while not hasattr(inner, "nc") and hasattr(inner, "model"):
    inner = inner.model
print(
    f"loaded {type(inner).__name__}: nc={inner.nc}, names={inner.names}, stride={inner.stride.tolist()}"
)
print(f"AutoShape conf={model.conf}, iou={model.iou}, max_det={model.max_det}")

In [ ]:
# a few random thermal frames from the test split (drone present); df paths are relative to DATA_ROOT
dataset = AntiUAVFrames(split="test", dataset_root=DATA_ROOT, drone_only=True)
print(f"{len(dataset)} test frame pairs with a drone; conditions: {dataset.condition_columns}")

N = 6
SEED = 7
idxs = dataset.df.sample(n=N, random_state=SEED).index.tolist()  # random frames, reproducible
samples = [dataset[i] for i in idxs]
for s in samples:
    gt = [round(v, 1) for v in s["box_thermal"].tolist()]
    print(f"  {s['sequence']} frame {s['frame_id']}: gt box_thermal (xywh) = {gt}")

In [ ]:
# predict on the thermal images; passing file paths lets the hub model do letterbox+NMS itself,
# and results.xyxy[i] comes back in the original 640x512 pixel coordinates
thermal_paths = [str(join(DATA_ROOT, s["thermal_path"])) for s in samples]

with torch.no_grad():
    results = model(thermal_paths)

for s, det in zip(samples, results.xyxy):
    gt = [round(v, 1) for v in s["box_thermal"].tolist()]
    print(f"\n{s['sequence']} frame {s['frame_id']}  gt(xywh)={gt}")
    print(f"  {s['thermal_path']}")
    if len(det) == 0:
        print("  no detections")
    for x1, y1, x2, y2, conf, cls in det.cpu().tolist():
        print(
            f"  pred xyxy=({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})  conf={conf:.3f}  cls={inner.names[int(cls)]}"
        )

In [ ]:
# visualize: GT box (green, xywh from the index) + predicted boxes (red, xyxy + conf) on the thermal frame
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

ncols = 2  # two columns, as many rows as the number of samples needs
nrows = math.ceil(len(samples) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
axes = fig.axes  # flat list of all subplot axes
for ax, s, det in zip(axes, samples, results.xyxy):
    img = Image.open(join(DATA_ROOT, s["thermal_path"]))
    ax.imshow(img, cmap="gray")
    gx, gy, gw, gh = s["box_thermal"].tolist()
    ax.add_patch(Rectangle((gx, gy), gw, gh, fill=False, edgecolor="lime", linewidth=2, label="gt"))
    for x1, y1, x2, y2, conf, cls in det.cpu().tolist():
        ax.add_patch(
            Rectangle(
                (x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2, label="pred"
            )
        )
        ax.text(x1, max(y1 - 6, 0), f"{inner.names[int(cls)]} {conf:.2f}", color="red", fontsize=9)
    ax.set_title(f"{s['sequence']} frame {s['frame_id']}")
    ax.legend(loc="upper left")
    ax.axis("off")
for ax in axes[len(samples) :]:  # hide unused subplots
    ax.axis("off")
plt.tight_layout()
plt.show()